# AI Cyber Threat Detection - Model Training

This notebook trains a simple machine learning model to classify network activity as normal or suspicious.

The goal is to build a basic AI-based cybersecurity detection pipeline using a public network intrusion dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_kddcup99
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
data = fetch_kddcup99(percent10=True, as_frame=True)
df = data.frame

df.head()

In [ ]:
# Convert the original labels into a binary target
# 0 = normal activity
# 1 = suspicious / attack activity

df["target"] = df["labels"].apply(lambda x: 0 if x == b"normal." else 1)

df["target"].value_counts()

In [ ]:
X = df.drop(columns=["labels", "target"])
y = df["target"]

X.head()

In [ ]:
# Use a sample to make training faster for a portfolio project

X_sample, _, y_sample, _ = train_test_split(
    X,
    y,
    train_size=100000,
    random_state=42,
    stratify=y
)

X_sample.shape, y_sample.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample
)

X_train.shape, X_test.shape

In [ ]:
categorical_features = X_train.select_dtypes(include=["object"]).columns
numeric_features = X_train.select_dtypes(exclude=["object"]).columns

categorical_features, numeric_features

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm

In [ ]:
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Normal", "Suspicious"])
plt.yticks([0, 1], ["Normal", "Suspicious"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

## Result

A Random Forest machine learning model was trained to classify network activity as either normal or suspicious.

This demonstrates a basic AI-based cybersecurity workflow:

1. Load network intrusion data
2. Prepare labels for binary classification
3. Train a machine learning model
4. Evaluate detection performance
5. Present the results clearly

This is a first version of the project and can later be improved with better feature engineering, anomaly detection models, explainability, and SOC-style alert summaries.